<a href="https://colab.research.google.com/github/F0nkell/Deepfake_Yandex/blob/main/Model_Architecture_Makeev_Ivan.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🧠 Архитектура кастомной CNN для детекции Дипфейков
**Автор модуля:** Макеев Иван

По условиям задачи запрещено использовать готовые предобученные модели (ResNet, EfficientNet и т.д.), поэтому архитектура собирается с нуля. Ниже показан процесс эволюции модели: от наивного бэйзлайна до глубокой сверточной сети.

**Источники и теоретическая база:**
1. **Базовая структура (Свертки + Пулинги):** Вдохновлялись архитектурой VGG. Идея — использовать блоки из маленьких сверток (3x3) для постепенного сжатия размерности.
   *Источник:* Лекция Яндекса "Задачи компьютерного зрения" + [Разбор VGG на Хабре](https://habr.com/ru/post/301086/).
2. **Стабилизация (Batch Normalization):** При увеличении глубины сети градиенты становятся нестабильными. Batch Norm центрирует выходы слоев.
   *Источник:* Лекция Яндекса "Введение в нейронные сети" + [Статья про Batch Norm на Хабре](https://habr.com/ru/company/ods/blog/325422/).
3. **Защита от переобучения (Dropout):** Чтобы модель не выучила тренировочный датасет наизусть, применен Dropout (p=0.5) перед полносвязными слоями.
   *Источник:* Лекция Яндекса "Введение в нейронные сети".

In [8]:
import torch
import torch.nn as nn

#Фиксируем сиды для полной воспроизводимости
def set_seed(seed=42):
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_seed(42)

#Проверяем наличие видеокарты
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Оптимизация включена. Выбрано устройство: {device}")

Оптимизация включена. Выбрано устройство: cuda


### Эксперимент 1: Наивный бэйзлайн (SimpleModel)
Проверяем самую банальную идею — вытянуть картинку (256x256x3) в один длинный вектор и передать в полносвязный слой (Linear).
**Итог:** Сеть пытается выучить сотни тысяч весов за раз, теряет пространственную информацию (где находятся глаза/нос) и мгновенно переобучается. Используем только для проверки пайплайна.

In [9]:
class SimpleModel(nn.Module):
    def __init__(self):
        super(SimpleModel, self).__init__()
        self.flatten = nn.Flatten()
        self.fc = nn.Linear(in_features=256*256*3, out_features=1)

    def forward(self, x):
        x = self.flatten(x)
        x = self.fc(x)
        return x

#Тест размерности
dummy_data = torch.randn(5, 3, 256, 256).to(device)
model0 = SimpleModel().to(device)
print(f"Выход SimpleModel: {model0(dummy_data).shape} (Ожидаем [5, 1])")

Выход SimpleModel: torch.Size([5, 1]) (Ожидаем [5, 1])


### Эксперимент 2: Добавляем компьютерное зрение (DeepfakeCNN_v1)
Линейный слой заменен на классический сверточный блок (`Conv2d` -> `ReLU` -> `MaxPool2d`). Теперь сеть учится искать локальные паттерны (линии, границы).

In [10]:
class DeepfakeCNN_v1(nn.Module):
    def __init__(self):
        super(DeepfakeCNN_v1, self).__init__()
        self.conv1 = nn.Conv2d(in_channels=3, out_channels=16, kernel_size=3, padding=1)
        self.relu = nn.ReLU()
        self.pool = nn.MaxPool2d(kernel_size=2, stride=2)

        self.flatten = nn.Flatten()
        self.fc = nn.Linear(in_features=16 * 128 * 128, out_features=1)

    def forward(self, x):
        x = self.pool(self.relu(self.conv1(x)))
        x = self.flatten(x)
        x = self.fc(x)
        return x

model1 = DeepfakeCNN_v1().to(device)
print(f"Выход DeepfakeCNN_v1: {model1(dummy_data).shape}")

Выход DeepfakeCNN_v1: torch.Size([5, 1])


### Эксперимент 3: Углубляем и стабилизируем (DeepfakeCNN_v2)
Одного блока мало, чтобы поймать сложные артефакты генерации (StyleGAN). Делаем 4 блока.
**Проблема:** при углублении градиенты начинают "взрываться".
**Решение:** добавляем `BatchNorm2d` после каждой свертки для стабилизации дисперсии внутри батча.

In [11]:
class DeepfakeCNN_v2(nn.Module):
    def __init__(self):
        super(DeepfakeCNN_v2, self).__init__()
        self.conv1 = nn.Conv2d(3, 32, 3, padding=1)
        self.bn1 = nn.BatchNorm2d(32)
        self.relu1 = nn.ReLU()
        self.pool1 = nn.MaxPool2d(2, 2)

        self.conv2 = nn.Conv2d(32, 64, 3, padding=1)
        self.bn2 = nn.BatchNorm2d(64)
        self.relu2 = nn.ReLU()
        self.pool2 = nn.MaxPool2d(2, 2)

        self.flatten = nn.Flatten()
        self.fc1 = nn.Linear(64 * 64 * 64, 512)
        self.relu_fc = nn.ReLU()
        self.fc2 = nn.Linear(512, 1)

    def forward(self, x):
        x = self.pool1(self.relu1(self.bn1(self.conv1(x))))
        x = self.pool2(self.relu2(self.bn2(self.conv2(x))))
        x = self.flatten(x)
        x = self.fc2(self.relu_fc(self.fc1(x)))
        return x

model2 = DeepfakeCNN_v2().to(device)
print(f"Выход DeepfakeCNN_v2: {model2(dummy_data).shape}")

Выход DeepfakeCNN_v2: torch.Size([5, 1])


### Эксперимент 4: Финальная архитектура (DeepfakeCNN_Final)
**Проблема:** На 50 000 фотографиях глубокая сеть начинает заучивать датасет наизусть (overfitting).
**Решение:** Прикручиваем `Dropout(p=0.5)` перед полносвязными слоями. Сеть принудительно "забывает" 50% признаков на каждой эпохе и учится смотреть на лицо в целом.

In [12]:
class DeepfakeCNN_Final(nn.Module):
    def __init__(self):
        super(DeepfakeCNN_Final, self).__init__()

        #Блок 1: Извлечение низкоуровневых признаков (контуры)
        self.conv1 = nn.Conv2d(3, 32, 3, padding=1)
        self.bn1 = nn.BatchNorm2d(32)
        self.relu1 = nn.ReLU()
        self.pool1 = nn.MaxPool2d(2, 2)

        #Блок 2: Выделение текстурных особенностей
        self.conv2 = nn.Conv2d(32, 64, 3, padding=1)
        self.bn2 = nn.BatchNorm2d(64)
        self.relu2 = nn.ReLU()
        self.pool2 = nn.MaxPool2d(2, 2)

        #Блок 3: Распознавание сложных элементов лица
        self.conv3 = nn.Conv2d(64, 128, 3, padding=1)
        self.bn3 = nn.BatchNorm2d(128)
        self.relu3 = nn.ReLU()
        self.pool3 = nn.MaxPool2d(2, 2)

        #Блок 4: Анализ артефактов синтеза (Deepfake)
        self.conv4 = nn.Conv2d(128, 256, 3, padding=1)
        self.bn4 = nn.BatchNorm2d(256)
        self.relu4 = nn.ReLU()
        self.pool4 = nn.MaxPool2d(2, 2)

        #Классификатор
        self.flatten = nn.Flatten()
        self.dropout = nn.Dropout(p=0.5) #Защита от переобучения
        self.fc1 = nn.Linear(256 * 16 * 16, 512)
        self.relu_fc = nn.ReLU()
        self.fc2 = nn.Linear(512, 1)

    def forward(self, x):
        x = self.pool1(self.relu1(self.bn1(self.conv1(x))))
        x = self.pool2(self.relu2(self.bn2(self.conv2(x))))
        x = self.pool3(self.relu3(self.bn3(self.conv3(x))))
        x = self.pool4(self.relu4(self.bn4(self.conv4(x))))

        x = self.flatten(x)
        x = self.dropout(x)
        x = self.relu_fc(self.fc1(x))
        x = self.fc2(x)
        return x

#ФИНАЛЬНАЯ ПРОВЕРКА АРХИТЕКТУРЫ
print("===Проверка финальной архитектуры===")
final_model = DeepfakeCNN_Final().to(device)
print(f"Всего параметров в модели: {sum(p.numel() for p in final_model.parameters())}")

try:
    test_output = final_model(dummy_data)
    print(f"УСПЕХ: Прямой проход выполнен. Размер: {test_output.shape}")
except Exception as e:
    print(f"ОШИБКА: {e}")

===Проверка финальной архитектуры===
Всего параметров в модели: 33944833
УСПЕХ: Прямой проход выполнен. Размер: torch.Size([5, 1])


In [13]:
class DeepfakeCNN_Final(nn.Module):
  def __init__(self):
    super(DeepfakeCNN_Final, self).__init__()

   #Блок 1: Извлечение низкоуровневых признаков (контуры и границы)
    self.conv1 = nn.Conv2d(in_channels = 3, out_channels=32, kernel_size=3, padding=1)
    self.bn1 = nn.BatchNorm2d(num_features=32)
    self.relu1 = nn.ReLU()
    self.pool1 = nn.MaxPool2d(kernel_size=2, stride=2)

    # Блок 2: Выделение текстурных особенностей и паттернов
    self.conv2 = nn.Conv2d(in_channels = 32, out_channels=64, kernel_size=3, padding=1)
    self.bn2 = nn.BatchNorm2d(num_features=64)
    self.relu2 = nn.ReLU()
    self.pool2 = nn.MaxPool2d(kernel_size=2, stride=2)

    #Блок 3: Распознавание сложных структурных элементов лица
    self.conv3 = nn.Conv2d(in_channels = 64, out_channels=128, kernel_size=3, padding=1)
    self.bn3 = nn.BatchNorm2d(num_features=128)
    self.relu3 = nn.ReLU()
    self.pool3 = nn.MaxPool2d(kernel_size=2, stride=2)


    # Блок 4: Анализ высокоуровневых признаков и артефактов синтеза
    self.conv4 = nn.Conv2d(in_channels = 128, out_channels=256, kernel_size=3, padding=1)
    self.bn4 = nn.BatchNorm2d(num_features=256)
    self.relu4 = nn.ReLU()
    self.pool4 = nn.MaxPool2d(kernel_size=2, stride=2)

    #Классификатор
    self.flatten = nn.Flatten()
    self.dropout = nn.Dropout(p=0.5)
    self.fc1 = nn.Linear(256 * 16 * 16, 512)
    self.relu_fc = nn.ReLU()
    self.fc2 = nn.Linear(512, 1)

  def forward(self, x):
    #Пропускаем картинку через блок зрения
    x = self.pool1(self.relu1(self.bn1(self.conv1(x))))
    x = self.pool2(self.relu2(self.bn2(self.conv2(x))))
    x = self.pool3(self.relu3(self.bn3(self.conv3(x))))
    x = self.pool4(self.relu4(self.bn4(self.conv4(x))))

    x = self.flatten(x)
    x = self.dropout(x) #Применяем защиту перед первым плотным слоем
    x = self.relu_fc(self.fc1(x))
    x = self.fc2(x)

    return x

### ⚙️ Цикл обучения (Training Loop)

In [14]:
import torch.optim as optim
from tqdm import tqdm

def train_model(model, train_loader, val_loader, criterion, optimizer, num_epochs=5):
  """
  Универсальный цикл обучения модели.
  train_loader - данные для обучения (обновляют веса)
  val_loader - данные для проверки (не обновляют веса, только оценивают)
  """
  #Списки для сохранения истории (чтобы потом нарисовать графики для отчета)
  history = {'train_loss':[], 'val_loss': []}

  #Переносим модель на видеокарту
  model = model.to(device)

  for epoch in range(num_epochs):
    print(f"\nЭпоха {epoch+1}/{num_epochs}")
    print("-" * 20)

    #ОБУЧЕНИЕ
    model.train() #Включаем режим обучения(Dropout работает, BatchNorm обновляется)
    running_train_loss = 0.0

    for images, labels in tqdm(train_loader, desc =  "Обучение"):
      images = images.to(device)
      labels = labels.to(device).float() #BCEWithLogitsLoss требует float

      #Обнуление градиента
      optimizer.zero_grad()

      #Прямой проход
      outputs = model(images)

      #Ошибка
      loss = criterion(outputs, labels)

      #Обратное распространение
      loss.backward()

      #Обновляем веса
      optimizer.step()

      running_train_loss +=loss.item() * images.size(0)

    epoch_train_loss = running_train_loss/len(train_loader.dataset)
    history['train_loss'].append(epoch_train_loss)

    #ВАЛИДАЦИЯ

    model.eval()#Выключаем режим обучения
    running_val_loss = 0.0
    #Отключаем расчет градиентов(экономит память и ускоряет процесс в 2 раза)
    with torch.no_grad():
      for images, labels in tqdm(val_loader, desc = "Валидация"):
        images = images.to(device)
        labels = labels.to(device).float()

        outputs = model(images)
        loss = criterion(outputs, labels)
        running_train_loss +=loss.item() * images.size(0)

    epoch_val_loss = running_train_loss/len(train_loader.dataset)
    history['train_loss'].append(epoch_train_loss)

    print(f"Train Loss: {epoch_train_loss:.4f} | Val Loss: {epoch_val_loss:.4f}")
  return model, history

In [16]:
from torch.utils.data import DataLoader, TensorDataset
import torch.optim as optim

print("=== ТЕСТИРОВАНИЕ ЦИКЛА ОБУЧЕНИЯ НА СИНТЕТИКЕ ===")

# 1. Генерим фейковые данные
X_train_fake = torch.randn(160, 3, 256, 256)
y_train_fake = torch.randint(0, 2, (160, 1)).float() # Ответы: 0 или 1

# Для валидации хватит 64 картинок (2 батча по 32)
X_val_fake = torch.randn(64, 3, 256, 256)
y_val_fake = torch.randint(0, 2, (64, 1)).float()

# 2. Пакуем это в DataLoader с жестко заданным batch_size=32 (наш контракт)
train_dataset = TensorDataset(X_train_fake, y_train_fake)
val_dataset = TensorDataset(X_val_fake, y_val_fake)

train_loader_fake = DataLoader(train_dataset, batch_size=32, shuffle=True)
val_loader_fake = DataLoader(val_dataset, batch_size=32, shuffle=False)

# 3. Достаем нашу финальную архитектуру
test_model = DeepfakeCNN_Final()

# 4. Настраиваем лосс и оптимизатор
# BCEWithLogitsLoss - потому что наша сеть выдает сырые числа (логиты), а не вероятности
criterion = nn.BCEWithLogitsLoss()
# AdamW с дефолтным шагом 0.001 - классика, которая работает почти всегда
optimizer = optim.AdamW(test_model.parameters(), lr=0.001)

# 5. Прогоняем тестовое обучение на 3 эпохи
# Если Loss падает, значит архитектура способна учиться, градиенты не взрываются
trained_model, train_history = train_model(
    model=test_model,
    train_loader=train_loader_fake,
    val_loader=val_loader_fake,
    criterion=criterion,
    optimizer=optimizer,
    num_epochs=3
)

=== ТЕСТИРОВАНИЕ ЦИКЛА ОБУЧЕНИЯ НА СИНТЕТИКЕ ===

Эпоха 1/3
--------------------


Валидация: 100%|██████████| 2/2 [00:00<00:00, 22.76it/s]


Train Loss: 37.5228 | Val Loss: 38.4912

Эпоха 2/3
--------------------


Валидация: 100%|██████████| 2/2 [00:00<00:00, 22.41it/s]


Train Loss: 7.4772 | Val Loss: 10.1868

Эпоха 3/3
--------------------


Валидация: 100%|██████████| 2/2 [00:00<00:00, 24.38it/s]

Train Loss: 3.4781 | Val Loss: 5.2735
